## Limpeza


## Setup

In [ ]:

dbutils.library.restartPython()
!pip install --upgrade pip
!pip install unidecode
!pip install rarfile
!pip install pdfplumber
!pip install nltk
!pip install openpyxl
!pip install xlrd
!pip install pytesseract
!pip install PyMuPDF
!pip install polars
%pip install dotenv

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, lit, regexp_replace, regexp_extract
import pyspark.sql.functions as F
from delta.tables import DeltaTable
from pyspark.sql.functions import current_timestamp
from pyspark.sql import Row
import pandas as pd
import os
import requests
import json
import datetime
from suporte.support_functions import *

from config import get_config
from config.secrets import get_sql_credentials

In [ ]:
dbutils.widgets.text("projeto", "")
projeto = dbutils.widgets.get("projeto").strip()
cfg = get_config(projeto)  # falha alto se `projeto` estiver vazio ou nao cadastrado

In [ ]:
output_filename = dbutils.jobs.taskValues.get(
    taskKey="fetch_from_aga_api",
    key="output_filename",
    debugValue=None
)

if output_filename is None:
    raise RuntimeError("Missing output_filename from task_1")

print("Received filename:", output_filename)

In [ ]:
# output_filename = 'apiCAMPO_20260501_20260827_20260828_1302'

In [ ]:
project_name = projeto
project_path = "/mnt/wst/" + project_name

# Define paths
folder_bronze = project_path + "/BRONZE/"
folder_silver = project_path + "/SILVER/"
folder_gold = project_path + "/GOLD/"
api_path = folder_bronze + "api/"

dbutils.fs.ls(api_path)

In [ ]:
# ------- SUMARIO DE EXECUCAO -------
execucao_steps = []

def log_step(etapa, df=None, status="Sucesso", observacoes="", registros_lidos=None, registros_escritos=None):
    """Registra uma etapa no sumario de execucao."""
    n = df.count() if df is not None else 0
    execucao_steps.append({
        "etapa": etapa,
        "status": status,
        "registros_lidos": registros_lidos if registros_lidos is not None else n,
        "registros_escritos": registros_escritos if registros_escritos is not None else n,
        "flags": _collect_flags(df) if df is not None else "\u2014",
        "observacoes": observacoes,
    })

def _collect_flags(df, colunas=None):
    from pyspark.sql import functions as F
    flag_cols = colunas if colunas is not None else [c for c in df.columns if c.startswith("flag_")]
    triggered = []
    for col_name in flag_cols:
        count = df.filter(F.col(col_name).isNotNull() & (F.col(col_name) != "")).count()
        if count > 0:
            triggered.append(f"{col_name}({count})")
    return "; ".join(triggered) if triggered else "\u2014"

def persistir_log():
    from pyspark.sql import Row
    import datetime as _dt
    try:
        notebook_name = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
        _execution_log_path = project_path + "/execution_log"
        _batch_ts = _dt.datetime.now().isoformat()
        _log_rows = [
            Row(
                batch_ts=_batch_ts,
                projeto=projeto,
                notebook=notebook_name.split("/")[-1],
                ordem=i,
                etapa=s["etapa"],
                status=s["status"],
                registros_lidos=int(s.get("registros_lidos") or 0),
                registros_escritos=int(s.get("registros_escritos") or 0),
                flags=str(s.get("flags") or "\u2014"),
                observacoes=str(s.get("observacoes") or ""),
                ts_registro=_dt.datetime.now().isoformat(),
            )
            for i, s in enumerate(execucao_steps)
        ]
        _log_df = spark.createDataFrame(_log_rows)
        _log_df.write.format("delta").mode("append").option("mergeSchema", "true").save(_execution_log_path)
        print(f"Log persistido: {len(_log_rows)} steps -> {_execution_log_path}")
    except Exception as _log_err:
        print(f"Aviso: falha ao persistir log de execucao \u2014 {_log_err}")

## 🔄 Load Latest Delta DataFrames

This **code block** automatically loads the most recent Delta DataFrames from the **API** folder located in the `BRONZE` layer (for the current `projeto`).

- If you want to use the most recent data, **leave the widgets empty**.
- If you want to load a previous version of the data, **enter the folder name** (e.g., `apiCAMPO_20220113_20220130_1435`) in the respective widget.


In [ ]:
from pyspark.sql import SparkSession
import re
# Obter valores dos widgets
api_folder = output_filename
# Funcao para encontrar o ultimo diretorio com padrao de data no nome
def get_latest_folder(base_path):
    folders = [f.name.rstrip("/") for f in dbutils.fs.ls(base_path) if f.isDir()]
    # Tenta extrair uma data do nome da pasta para ordenacao
    def extract_date_key(name):
        match = re.search(r"(\d{8}(_\d{4})?)", name)
        return match.group(1) if match else "00000000_0000"
    sorted_folders = sorted(folders, key=extract_date_key, reverse=True)
    return sorted_folders[0] if sorted_folders else None

# Determinar caminho da API
if api_folder:
    full_api_path = api_path + api_folder
else:
    latest_api_folder = get_latest_folder(api_path)
    full_api_path = api_path + latest_api_folder if latest_api_folder else None

# Ler o DataFrame da API
sem_dados = False
try:
    if full_api_path:
        print(f"📁 Reading API data from: {full_api_path}")
        df_api = spark.read.format("delta").load(full_api_path)

        execucao_steps.append({
            "etapa": "Leitura da API",
            "status": "Sucesso",
            "registros_lidos": df_api.count(),
            "registros_escritos": df_api.count(),
            "flags": "\u2014",
            "observacoes": f"Dados carregados da Bronze em {full_api_path}"
        })
    else:
        print("⚠️ No API folder found or selected.")
        execucao_steps.append({
            "etapa": "Leitura da API",
            "status": "Aviso",
            "registros_lidos": 0,
            "registros_escritos": 0,
            "flags": "\u2014",
            "observacoes": "Nenhuma pasta de API encontrada \u2014 encerrando sem processar"
        })
        sem_dados = True
except Exception as _e:
    execucao_steps.append({
        "etapa": "Leitura da API", "status": "Erro",
        "registros_lidos": 0, "registros_escritos": 0, "flags": "\u2014", "observacoes": str(_e)
    })
    persistir_log()
    raise

if sem_dados:
    persistir_log()
    dbutils.notebook.exit("Nenhuma pasta de API encontrada")

In [ ]:
display(pd.DataFrame(execucao_steps))

### Limpeza

Conferencia data

In [ ]:
from pyspark.sql.functions import to_timestamp, coalesce, col

# Define os formatos que voce quer tentar
formatos_data = [
    "dd/MM/yyyy HH:mm",
    "yyyy-MM-dd HH:mm:ss",
    "dd-MM-yy HH:mm",
    "MM-dd-yy HH:mm",
    "yyyy-MM-dd'T'HH:mm:ss.SSS'Z'"
]
# Lista das colunas de data
colunas_data = ["dataHoraAmostragem", "dataRecebLab", "dataEnvioLab", "dataHoraAnalise", "dataLiberacao"]

# Para cada coluna de data, cria uma versao convertida chamada <coluna>_valida
for coluna in colunas_data:
    # Constroi a conversao com multiplos formatos usando coalesce
    coluna_convertida = coalesce(
        *[to_timestamp(col(coluna), fmt) for fmt in formatos_data]
    )
    # Adiciona a nova coluna ao DataFrame
    df_api = df_api.withColumn(f"{coluna}_valida", coluna_convertida)
    df_api = df_api.withColumn(
       f"flag_{coluna}",
        F.when(
            col(f"{coluna}_valida").isNull(),
            F.lit(f"falha na conversao de {coluna}")
        ).otherwise(F.lit(None).cast("string"))
    )

In [ ]:
# Soma as falhas de todas as colunas de data, nao so uma
falhas_totais = sum(
    df_api.filter(F.col(f"flag_{coluna}").isNotNull()).count()
    for coluna in colunas_data
)
execucao_steps.append({
    "etapa": "Normalizacao de datas",
    "status": "Sucesso" if falhas_totais == 0 else "Aviso",
    "registros_lidos": df_api.count(),
    "registros_escritos": df_api.count(),
    "flags": _collect_flags(df_api, [f"flag_{coluna}" for coluna in colunas_data]),
    "observacoes": f"Falhas de timestamp: {falhas_totais}"
})

In [ ]:
display(pd.DataFrame(execucao_steps))

### Limpeza Matriz

In [ ]:
df_api.display()

In [ ]:
from pyspark.sql import functions as F

# --- Normalizacao do valor de "matriz" (Agua Subterranea -> sem acentuacao, pra subir certo no banco) ---
# Feito bem no inicio: "matriz" alimenta o join com o escopo, o Resumo por Station, samp_matrix no
# Excel final etc. -- normalizando aqui, tudo que vem depois ja usa o valor corrigido automaticamente.
df_api = df_api.withColumn(
    "matriz",
    F.when(F.col("matriz") == "Água Subterrânea", F.lit("Agua Subterranea")).otherwise(F.col("matriz"))
)

### Mapeamento da Sample Name

In [ ]:
# Credenciais compartilhadas entre projetos; so o filtro `source_project` muda por projeto
sql_creds = get_sql_credentials()

jdbcUrl = (
    f"jdbc:sqlserver://{sql_creds['host']}:{sql_creds['port']};"
    f"databaseName={sql_creds['database']};encrypt=false;trustServerCertificate=true;"
)

query_station = f"""
select
    s.Name as codigoHga,
    s.alternate_name,
    smp.Name as SampleName
from station s
join SYS_Sample smp
    on s.ID = smp.Station
where source_project = '{cfg["source_project_sql"]}'
"""

df_station = spark.read.format("jdbc") \
    .option("url", jdbcUrl) \
    .option("query", query_station) \
    .option("user", sql_creds["user"]) \
    .option("password", sql_creds["password"]) \
    .load()


In [ ]:
from pyspark.sql.functions import col, regexp_replace, trim

df_api = df_api.withColumn(
    "descricaoAmostra",
    regexp_replace(regexp_replace(col("descricaoAmostra"), "_", "-"), r"\s+", "")
)

df_api = df_api.withColumn(
    "descricaoAmostra",
    regexp_replace(col("descricaoAmostra"), r"\.", ",")
)

# df_api.select("descricaoAmostra").display()

In [ ]:
from pyspark.sql.functions import col, regexp_replace
df_api = df_api.withColumn(
    'descricaoAmostra',
    regexp_replace(col('descricaoAmostra'), '-Terc', '')
)

In [ ]:
df_api = df_api.drop('codigoHga')

In [ ]:
df_api = df_api.join(
    df_station,
    df_api["descricaoAmostra"] == df_station["SampleName"],
    "left"
).drop('SampleName').withColumn(
    "flag_station_sem_mapeamento",
    F.when(F.col("alternate_name").isNull(), F.lit("Amostra sem correspondencia em df_station"))
    .otherwise(F.lit(None).cast("string"))
)

## Correcao Manual do sample_name

In [ ]:
from pyspark.sql import functions
import pandas as pd
import openpyxl
from sharepoint_connector import download_file_by_name

depara_local_path = f"/tmp/{cfg['depara_filename']}"

# ===== 1. EXTRAIR CASOS SEM CORRESPONDENCIA (distintos) =====

df_novos = (
    df_api
    .filter(functions.col("flag_station_sem_mapeamento").isNotNull())
    .select("descricaoAmostra", "matriz")
    .distinct()
)
pdf_novos = df_novos.toPandas()

# ===== 2. BAIXAR O DE-PARA ATUAL DO SHAREPOINT =====

result = download_file_by_name(
    folder_path=cfg["sharepoint_depara_folder"],
    filename=cfg["depara_filename"],
    local_path=depara_local_path
)
if result["status"] != "success":
    raise Exception(f"Erro ao baixar mapeamento: {result['error']}")


In [ ]:
# ===== 3. ACRESCENTAR SO OS CASOS NOVOS, SEM PERDER O QUE JA FOI MAPEADO =====
wb = openpyxl.load_workbook(depara_local_path)
ws = wb["DeparaSampleName"]

existentes = {
    row[0].value for row in ws.iter_rows(min_row=2, max_col=1) if row[0].value is not None
}

pdf_para_adicionar = pdf_novos[~pdf_novos["descricaoAmostra"].isin(existentes)]

for _, linha in pdf_para_adicionar.iterrows():
    ws.append([linha["descricaoAmostra"], linha["matriz"], None, None, None])

wb.save(depara_local_path)


In [ ]:
# ===== 4. LER O DE-PARA JA CORRIGIDO =====

mapping_pd = pd.read_excel(depara_local_path, sheet_name="DeparaSampleName", dtype=str)
mapping_pd_nome = mapping_pd[mapping_pd["descricaoAmostra_para"].notna()]

mapping_sample_name = spark.createDataFrame(
    mapping_pd_nome[["descricaoAmostra_de", "descricaoAmostra_para", "codigoHga", "alternate_name"]]
).withColumnRenamed("codigoHga", "codigoHga_depara") \
 .withColumnRenamed("alternate_name", "alternate_name_depara")

# ===== 5. APLICAR A CORRECAO DE descricaoAmostra =====

df_api = (
    df_api
    .join(
        mapping_sample_name,
        df_api["descricaoAmostra"] == mapping_sample_name["descricaoAmostra_de"],
        "left"
    )
    .withColumn(
        "descricaoAmostra_final",
        functions.coalesce(functions.col("descricaoAmostra_para"), functions.col("descricaoAmostra"))
    )
    .drop("descricaoAmostra_de", "descricaoAmostra_para")
)

# ===== 6. REFAZER O JOIN ORIGINAL COM df_station (sem sobrescrever o que ja esta certo) =====

df_station_join = (
    df_station
    .withColumnRenamed("codigoHga", "codigoHga_station")
    .withColumnRenamed("alternate_name", "alternate_name_station")
)

df_api = (
    df_api.join(
        df_station_join,
        df_api["descricaoAmostra_final"] == df_station_join["SampleName"],
        "left"
    )
    .drop("SampleName")
    .withColumn(
        "flag_station_sem_mapeamento",
        functions.when(functions.col("alternate_name_station").isNull(), functions.lit("Amostra sem correspondencia em df_station"))
        .otherwise(functions.lit(None).cast("string"))
    )
)

# ===== 7. PREENCHER codigoHga E alternate_name SO ONDE ESTAVA VAZIO =====
# prioridade: valor original (ja correto) > valor do rejoin corrigido > valor manual do de-para

df_api = (
    df_api
    .withColumn(
        "codigoHga",
        functions.coalesce(functions.col("codigoHga"), functions.col("codigoHga_station"), functions.col("codigoHga_depara"))
    )
    .withColumn(
        "alternate_name",
        functions.coalesce(functions.col("alternate_name"), functions.col("alternate_name_station"), functions.col("alternate_name_depara"))
    )
    .drop("codigoHga_station", "alternate_name_station", "codigoHga_depara", "alternate_name_depara")
)
# ===== 8. RECALCULAR A FLAG COM BASE NO VALOR FINAL (JOIN OU OVERRIDE MANUAL) =====

df_api = df_api.withColumn(
    "flag_station_sem_mapeamento",
    functions.when(functions.col("alternate_name").isNull(), functions.lit("Amostra sem correspondencia em df_station"))
    .otherwise(functions.lit(None).cast("string"))
)

In [ ]:
df_api.select('codigoHga','alternate_name','descricaoAmostra_final','flag_station_sem_mapeamento').distinct().display()

In [ ]:
df_api = df_api.withColumn(
    "profInicial",
    regexp_extract(col('descricaoAmostra_final'), r"([\d,]+)$", 1)
)

df_api = df_api.withColumn(
    "profInicial",
    regexp_replace(col("profInicial"), ",", ".").cast("double")
)

In [ ]:
from pyspark.sql.functions import col, when, regexp_replace, trim

df_api = df_api.withColumn(
    "resultadoNumerico",
    regexp_replace(col("resultadoNumerico"), ",", ".").cast("double")
)

### Mapeamento Depara (Parametros)

In [ ]:
from sharepoint_connector import download_file_by_name
import pandas as pd
# ===== DOWNLOAD DE MAPEAMENTO DOS PARAMETROS DO SHAREPOINT =====
try:
    result = download_file_by_name(
        folder_path=cfg["sharepoint_depara_folder"],
        filename=cfg["depara_filename"],
        local_path=depara_local_path
    )
    if result["status"] != "success":
        raise Exception(f"Erro ao baixar mapeamento: {result['error']}")

    # Carregar em pandas e depois em Spark
    mapping_pd_1 = pd.read_excel(depara_local_path, sheet_name="DeparaParametros", dtype=str)
    mapping_df = spark.createDataFrame(mapping_pd_1)

    execucao_steps.append({
        "etapa": "Download mapeamento (SharePoint)",
        "status": "Sucesso",
        "registros_lidos": mapping_df.count(),
        "registros_escritos": mapping_df.count(),
        "flags": "\u2014",
        "observacoes": f"{cfg['depara_filename']} baixado e carregado"
    })
except Exception as _e:
    execucao_steps.append({
        "etapa": "Download mapeamento (SharePoint)", "status": "Erro",
        "registros_lidos": 0, "registros_escritos": 0, "flags": "\u2014", "observacoes": str(_e)
    })
    persistir_log()
    raise

In [ ]:
df_api = df_api.withColumn('parametroCorrigido', col("parametroOriginal"))

In [ ]:
df_api = df_api.withColumn('flag_padronizacao', F.lit(None).cast("string"))
try:
    mapping_df_selected = mapping_df.select("parametro_de", "matriz_de", "parametro_para", 'unidade_para')

    df_api = (
        df_api
        .join(
            mapping_df_selected,
            (df_api["parametroCorrigido"] == mapping_df_selected["parametro_de"]) &
            (df_api["matriz"] == mapping_df_selected["matriz_de"]),
            how="left"
        )
        .withColumn("parametroCorrigido_original", F.col("parametroCorrigido"))
        .withColumn("parametroCorrigido", F.col("parametro_para"))
        .withColumn(
            "flag_padronizacao",
            F.when(
                F.col("parametro_para").isNull() | (F.col("parametro_para") == ""),
                F.lit("Parametro nao mapeado")
            ).otherwise(F.col("flag_padronizacao"))
        )
        .drop("parametro_de", "matriz_de", "parametro_para")
    )

    # Detalhe: quais parametros e matrizes especificos nao foram mapeados
    parametros_nao_mapeados = [
        f'{row["parametroCorrigido_original"]} ({row["matriz"]})'
        for row in df_api.filter(F.col("flag_padronizacao") == "Parametro nao mapeado")
                          .select("parametroCorrigido_original", "matriz")
                          .distinct()
                          .collect()
    ]
    falhas = df_api.filter(F.col("flag_padronizacao").isNotNull()).count()

    execucao_steps.append({
        "etapa": "Mapeamento de parametros",
        "status": "Sucesso" if falhas == 0 else "Aviso",
        "registros_lidos": df_api.count(),
        "registros_escritos": df_api.count(),
        "flags": _collect_flags(df_api, ["flag_padronizacao"]),
        "observacoes": (
            "Todos os parametros mapeados com sucesso" if falhas == 0
            else f"Parametros nao mapeados ({falhas} registros): {', '.join(parametros_nao_mapeados)}"
        )
    })

except Exception as _e:
    execucao_steps.append({
        "etapa": "Mapeamento de parametros", "status": "Erro",
        "registros_lidos": 0, "registros_escritos": 0, "flags": "\u2014", "observacoes": str(_e)
    })
    persistir_log()
    raise

In [ ]:
display(pd.DataFrame(execucao_steps))

### Conversao de unidades

In [ ]:
df_api.display()

In [ ]:
from functools import reduce

fatores_conversao = {
    ("%", "mg/kg"): 10000,     # 1% = 10000 mg/kg
    ("mg/dm³", "mg/kg"): 1,
    ('-', 'unit'): 1,
    ('µg/L', 'mg/L'): 0.001,
    ('mg CaCO3/L', 'mg/L'): 1,
    ('NMP/100mL', 'P/A'): 1,
    ('mgO2/L', 'mg/L'): 1
}

itens_fatores = list(fatores_conversao.items())

try:
    fator_conversao = reduce(
        lambda acc, kv: acc.when(
            (F.col("unidadeOriginal") == kv[0][0]) & (F.col("unidade_para") == kv[0][1]),
            F.lit(kv[1])
        ),
        itens_fatores[1:],
        F.when(
            (F.col("unidadeOriginal") == itens_fatores[0][0][0]) & (F.col("unidade_para") == itens_fatores[0][0][1]),
            F.lit(itens_fatores[0][1])
        )
    ).otherwise(F.lit(1))

    # Mesma logica do fator_conversao, mas devolve True/False -> indica se o par de unidades esta mapeado
    fator_encontrado = reduce(
        lambda acc, kv: acc.when(
            (F.col("unidadeOriginal") == kv[0][0]) & (F.col("unidade_para") == kv[0][1]),
            F.lit(True)
        ),
        itens_fatores[1:],
        F.when(
            (F.col("unidadeOriginal") == itens_fatores[0][0][0]) & (F.col("unidade_para") == itens_fatores[0][0][1]),
            F.lit(True)
        )
    ).otherwise(F.lit(False))

    df_api = (
        df_api
        .withColumn("fator_encontrado", fator_encontrado)
        .withColumn(
            "resultadoCorrigido",
            F.when(
                F.col("unidadeOriginal") != F.col("unidade_para"),
                F.round(F.col("resultadoNumerico") * fator_conversao, 4)
            ).otherwise(F.col("resultadoNumerico"))
        )
        .withColumn(
            "flag_conversao",
            F.when(
                F.col("unidadeOriginal").isNotNull() & F.col("unidade_para").isNotNull() &
                (F.col("unidadeOriginal") != F.col("unidade_para")) & F.col("fator_encontrado"),
                F.concat(F.lit("Valor convertido de "), F.col("unidadeOriginal"), F.lit(" para "), F.col("unidade_para"))
            ).otherwise(F.lit(None).cast("string"))
        )
        .withColumn(
            "flag_fator_nao_encontrado",
            F.when(
                F.col("unidadeOriginal").isNotNull() & F.col("unidade_para").isNotNull() &
                (F.col("unidadeOriginal") != F.col("unidade_para")) & (~F.col("fator_encontrado")),
                F.concat(F.lit("Fator de conversao nao encontrado: "), F.col("unidadeOriginal"), F.lit(" -> "), F.col("unidade_para"))
            ).otherwise(F.lit(None).cast("string"))
        )
    )

    # Detalhe: parametro, matriz, unidade que veio e unidade esperada, quando o fator nao existe no mapa
    conversoes_sem_fator = [
        f'{row["parametroCorrigido_original"]} ({row["matriz"]}): veio em "{row["unidadeOriginal"]}", esperado "{row["unidade_para"]}"'
        for row in df_api.filter(F.col("flag_fator_nao_encontrado").isNotNull())
                          .select("parametroCorrigido_original", "matriz", "unidadeOriginal", "unidade_para")
                          .distinct()
                          .collect()
    ]

    df_api = df_api.drop("fator_encontrado")

    falhas_conversao = df_api.filter(F.col("flag_fator_nao_encontrado").isNotNull()).count()

    execucao_steps.append({
        "etapa": "Conversao de unidades",
        "status": "Sucesso" if falhas_conversao == 0 else "Aviso",
        "registros_lidos": df_api.count(),
        "registros_escritos": df_api.count(),
       "flags": _collect_flags(df_api, ["flag_conversao", "flag_fator_nao_encontrado"]),
        "observacoes": (
            "Todas as conversoes de unidade aplicadas com sucesso" if falhas_conversao == 0
            else f"Fator de conversao nao encontrado ({falhas_conversao} registros): {'; '.join(conversoes_sem_fator)}"
        )
    })
except Exception as _e:
    execucao_steps.append({
        "etapa": "Conversao de unidades", "status": "Erro",
        "registros_lidos": 0, "registros_escritos": 0, "flags": "\u2014", "observacoes": str(_e)
    })
    persistir_log()
    raise


## 💾 Save API to Delta Lake

In [ ]:
import re

def clean_column_names(df):
    for col_name in df.columns:
        clean_name = re.sub(r"[ ,;{}\(\)\n\t=]", "_", col_name)  # Substitui por underscore
        df = df.withColumnRenamed(col_name, clean_name)
    return df

# Aplicar a limpeza
df_api_clean = clean_column_names(df_api)

In [ ]:
folder_silver_validado = project_path + "/SILVER/api_limpo"
try:
    df_api_clean.write.format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .save(folder_silver_validado)

    execucao_steps.append({
        "etapa": "Gravacao Silver",
        "status": "Sucesso",
        "registros_lidos": df_api_clean.count(),
        "registros_escritos": df_api_clean.count(),
        "flags": "\u2014",
        "observacoes": f"Salvo em {folder_silver}"
    })
except Exception as _e:
    execucao_steps.append({
        "etapa": "Gravacao Silver",
        "status": "Erro",
        "registros_lidos": df_api_clean.count(),
        "registros_escritos": 0,
        "flags": "\u2014",
        "observacoes": str(_e)
    })
    persistir_log()
    raise

In [ ]:
# ------- PERSISTIR LOG DE EXECUCAO -------
persistir_log()
import pandas as pd
display(pd.DataFrame(execucao_steps))